In [1]:
# ============================================================
# PS S6E5 - exp_20260516_033_blend_logreg_topk_feature_variants_min
#
# D: blend-code small improvements
# - Avg / HC / NM / Signed SLSQP on raw OOF predictions
# - Prune weak models
# - Ridge / LogReg on current raw+rank+logit meta features
# - Additional LogReg feature variants: logit_only, raw_logit, raw_only
# - Additional LogReg topK equal blends from C-search candidates
# - Additional Ridge feature variants: logit_only, raw_logit
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260516_033_blend_logreg_topk_feature_variants_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================
#
# D experiment stack:
# - Drop 030/031 because shared011 CatBoost meta-stacking did not improve Public LB.
# - Drop old RealMLP variants 007/022/023/027.
# - Drop 018 by default because recent optimized blends effectively zeroed it out after 029/032.
# - Keep 014/015/016/017/021/026/028/029/032.
#
# If you want a conservative comparison, re-add 018 manually and rerun.
# ============================================================

MODEL_SPECS = [
    {
        "key": "014",
        "name": "cat_shared004_no_tyreageratio_str_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "pred": "pred_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "public_lb": 0.95233,
    },
    {
        "key": "015",
        "name": "cat_shared004_no_race_year_groupstats_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "pred": "pred_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "public_lb": 0.95244,
    },
    {
        "key": "016",
        "name": "xgb_shared005_fe_te_seedens_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "pred": "pred_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "017",
        "name": "xgb_shared005_no_pitstop_pairte_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "pred": "pred_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "021",
        "name": "tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min",
        "family": "TabM",
        "oof": "oof_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "pred": "pred_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "public_lb": 0.95171,
    },
    {
        "key": "026",
        "name": "lgb_weather_year_stint_flags_min",
        "family": "LightGBM",
        "oof": "oof_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
        "pred": "pred_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
        "public_lb": 0.95096,
    },
    {
        "key": "028",
        "name": "parent_child_mlp_shared010_min",
        "family": "Parent-Child MLP",
        "oof": "oof_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "pred": "pred_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "public_lb": 0.95260,
    },
    {
        "key": "029",
        "name": "realmlp_shared001v2_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260514_029_realmlp_shared001v2_min.npy",
        "pred": "pred_exp_20260514_029_realmlp_shared001v2_min.npy",
        "public_lb": 0.95397,
    },
    {
        "key": "032",
        "name": "custom_realmlp_cv095352_min",
        "family": "CustomTorchRealMLP",
        "oof": "oof_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "pred": "pred_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "public_lb": 0.95309,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("model_keys:", model_keys)
print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)


model_keys: ['014', '015', '016', '017', '021', '026', '028', '029', '032']
X_raw: (439140, 9)
T_raw: (188165, 9)


In [7]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
7,029,realmlp_shared001v2_min,RealMLP,0.954042,0.95397,4.969082e-08,0.998981,0.000008,0.998439
8,032,custom_realmlp_cv095352_min,CustomTorchRealMLP,0.953506,0.95309,3.416236e-04,0.998800,0.000474,0.998323
0,014,cat_shared004_no_tyreageratio_str_min,CatBoost,0.952726,0.95233,7.059056e-05,0.997215,0.000153,0.996355
1,015,cat_shared004_no_race_year_groupstats_min,CatBoost,0.952714,0.95244,4.489366e-05,0.997058,0.000141,0.996636
2,016,xgb_shared005_fe_te_seedens_min,XGBoost,0.951986,0.95164,1.674493e-05,0.997698,0.000027,0.997298
3,017,xgb_shared005_no_pitstop_pairte_min,XGBoost,0.951939,0.95164,1.331864e-05,0.997702,0.000024,0.997621
4,021,tabm_shared007_comp_oof_no_pitstop_no_isorigin...,TabM,0.951492,0.95171,1.000000e-06,0.999431,0.000003,0.996751
6,028,parent_child_mlp_shared010_min,Parent-Child MLP,0.951374,0.95260,2.206709e-04,0.994038,0.000300,0.992415
5,026,lgb_weather_year_stint_flags_min,LightGBM,0.951279,0.95096,1.470362e-05,0.999081,0.000018,0.998904


best single: 029 0.9540420784720796


In [8]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,014,015,016,017,021,026,028,029,032
014,1.000000,0.996917,0.962898,0.962739,0.968765,0.984963,0.956846,0.961507,0.960927
015,0.996917,1.000000,0.962398,0.962237,0.967914,0.984951,0.956020,0.960397,0.959915
016,0.962898,0.962398,1.000000,0.998485,0.978284,0.960326,0.972366,0.980101,0.979115
017,0.962739,0.962237,0.998485,1.000000,0.978150,0.960937,0.972213,0.979812,0.979023
021,0.968765,0.967914,0.978284,0.978150,1.000000,0.966370,0.974491,0.980413,0.982081
026,0.984963,0.984951,0.960326,0.960937,0.966370,1.000000,0.954081,0.957061,0.958169
028,0.956846,0.956020,0.972366,0.972213,0.974491,0.954081,1.000000,0.981320,0.979124
029,0.961507,0.960397,0.980101,0.979812,0.980413,0.957061,0.981320,1.000000,0.990775
032,0.960927,0.959915,0.979115,0.979023,0.982081,0.958169,0.979124,0.990775,1.000000


Spearman correlation


,014,015,016,017,021,026,028,029,032
014,1.000000,0.994676,0.975612,0.975146,0.970041,0.973115,0.968397,0.976608,0.972390
015,0.994676,1.000000,0.975249,0.974943,0.969630,0.972735,0.968181,0.976407,0.972607
016,0.975612,0.975249,1.000000,0.998203,0.965983,0.969232,0.967417,0.972249,0.966370
017,0.975146,0.974943,0.998203,1.000000,0.965901,0.969808,0.967124,0.972103,0.966202
021,0.970041,0.969630,0.965983,0.965901,1.000000,0.962785,0.967054,0.967894,0.966383
026,0.973115,0.972735,0.969232,0.969808,0.962785,1.000000,0.959946,0.966439,0.965142
028,0.968397,0.968181,0.967417,0.967124,0.967054,0.959946,1.000000,0.973048,0.967367
029,0.976608,0.976407,0.972249,0.972103,0.967894,0.966439,0.973048,1.000000,0.980598
032,0.972390,0.972607,0.966370,0.966202,0.966383,0.965142,0.967367,0.980598,1.000000


In [9]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T, keys):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in keys] +
        [f"rank_{k}" for k in keys] +
        [f"logit_{k}" for k in keys]
    )

    return X_meta, T_meta, feature_names


X_meta_full, T_meta_full, meta_feature_names_full = build_meta_features(
    X_raw,
    T_raw,
    model_keys,
)

print("X_meta_full:", X_meta_full.shape)
print("T_meta_full:", T_meta_full.shape)
print("meta meta_feature_names_full:", meta_feature_names_full)

def build_meta_features_variant(X, T, keys, variant="raw_rank_logit"):
    """Build meta features with explicit feature-set variant.

    Variants:
      - raw_rank_logit: current baseline
      - raw_logit
      - logit_only
      - raw_only
      - rank_only
      - raw_rank
      - rank_logit
    """
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    parts_X = []
    parts_T = []
    names = []

    def add_part(prefix, X_part, T_part):
        parts_X.append(X_part)
        parts_T.append(T_part)
        names.extend([f"{prefix}_{k}" for k in keys])

    if variant == "raw_rank_logit":
        add_part("raw", X, T)
        add_part("rank", X_rank, T_rank)
        add_part("logit", X_logit, T_logit)
    elif variant == "raw_logit":
        add_part("raw", X, T)
        add_part("logit", X_logit, T_logit)
    elif variant == "logit_only":
        add_part("logit", X_logit, T_logit)
    elif variant == "raw_only":
        add_part("raw", X, T)
    elif variant == "rank_only":
        add_part("rank", X_rank, T_rank)
    elif variant == "raw_rank":
        add_part("raw", X, T)
        add_part("rank", X_rank, T_rank)
    elif variant == "rank_logit":
        add_part("rank", X_rank, T_rank)
        add_part("logit", X_logit, T_logit)
    else:
        raise ValueError(f"Unknown meta feature variant: {variant}")

    return np.column_stack(parts_X), np.column_stack(parts_T), names


X_meta_full: (439140, 27)
T_meta_full: (188165, 27)
meta meta_feature_names_full: ['raw_014', 'raw_015', 'raw_016', 'raw_017', 'raw_021', 'raw_026', 'raw_028', 'raw_029', 'raw_032', 'rank_014', 'rank_015', 'rank_016', 'rank_017', 'rank_021', 'rank_026', 'rank_028', 'rank_029', 'rank_032', 'logit_014', 'logit_015', 'logit_016', 'logit_017', 'logit_021', 'logit_026', 'logit_028', 'logit_029', 'logit_032']


In [10]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
    weight_keys=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "weight_keys": None if weights is None else list(weight_keys or model_keys),
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        f"delta_vs_{best_single_key}": float(score - auc(y, oofs[best_single_key])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [11]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    # "avg_007_014_015_016_018_020_021_022_023": ["007", "014", "015", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_016_018_020_021_022_023": ["007", "014", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_015_018_020_021_022_023": ["007", "014", "015", "018", "020", "021", "022", "023"],
    "avg" : model_keys
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg: 0.954509755


In [12]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.954809069
HC weights:
014 0.06064719601920769
015 0.048587361502969974
016 0.1103919823899879
017 0.025639845806883402
021 0.03595406092182361
026 0.0538791309276566
028 0.07219352191115204
029 0.4018386845209314
032 0.19086821599938733


In [13]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.954809
         Iterations: 190
         Function evaluations: 428
nm_softmax_raw: 0.954809106
NM weights:
014 0.05978282579359779
015 0.04907271132859998
016 0.11107764886230032
017 0.02478251474173071
021 0.03609762256140798
026 0.05358008519235927
028 0.07113406093719117
029 0.4032975741398246
032 0.1911749564429881


In [14]:
# ============================================================
# Signed SLSQP weights
# Allow small negative weights, with sum(w)=1
# ============================================================

def weighted_average_signed(X, w):
    return np.asarray(X, dtype=float) @ np.asarray(w, dtype=float)


def optimize_signed_slsqp(
    X,
    y,
    start_w=None,
    neg_bound=-0.10,
    pos_bound=0.60,
    maxiter=1000,
):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n
    else:
        start_w = np.asarray(start_w, dtype=float)

    # Start must satisfy sum=1
    start_w = start_w / start_w.sum()

    bounds = [(neg_bound, pos_bound) for _ in range(n)]

    constraints = [
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1.0,
        }
    ]

    def objective(w):
        p = weighted_average_signed(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        start_w,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": maxiter,
            "ftol": 1e-12,
            "disp": True,
        },
    )

    w = res.x
    score = auc(y, weighted_average_signed(X, w))

    return w, score, res

In [15]:
print("\n" + "=" * 80)
print("Signed SLSQP")
print("=" * 80)

signed_w, signed_score, signed_res = optimize_signed_slsqp(
    X_raw,
    y,
    start_w=nm_w,
    neg_bound=-0.05,
    pos_bound=0.50,
    maxiter=1000,
)

add_candidate(
    name="slsqp_signed_raw_neg005",
    method="slsqp_signed",
    oof_pred=weighted_average_signed(X_raw, signed_w),
    test_pred=weighted_average_signed(T_raw, signed_w),
    weights=signed_w,
    params={
        "constraint": "sum1_signed",
        "neg_bound": -0.05,
        "pos_bound": 0.50,
        "success": bool(signed_res.success),
        "message": str(signed_res.message),
        "nit": int(getattr(signed_res, "nit", -1)),
        "fun": float(signed_res.fun),
    },
    notes="signed SLSQP on raw OOF predictions; high overfit risk; attack-only",
)

print("Signed SLSQP weights:")
for k, w in zip(model_keys, signed_w):
    print(k, w)

print("Signed score:", signed_score)
print("sum weights:", signed_w.sum())
print("min weight:", signed_w.min())
print("max weight:", signed_w.max())


Signed SLSQP
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.9548090746724864
            Iterations: 18
            Function evaluations: 341
            Gradient evaluations: 18
slsqp_signed_raw_neg005: 0.954809075
Signed SLSQP weights:
014 0.05978210170397174
015 0.0491951917885479
016 0.11113338264130986
017 0.024554214258865753
021 0.036138239922739505
026 0.05361434896920977
028 0.07109469305954727
029 0.40340941860291124
032 0.1910784090528971
Signed score: 0.9548090746724864
sum weights: 1.0000000000000002
min weight: 0.024554214258865753
max weight: 0.40340941860291124


In [16]:
# ============================================================
# Prune models based on HC / NM / SLSQP weights
# ============================================================

PRUNE_HARD_THRESHOLD = 0.005
PRUNE_SOFT_THRESHOLD = 0.020
KEEP_PROTECTED_KEYS = {
    "014", "015",  # CatBoost branch
    "016", "017",  # XGB branch; 017 is historically useful for LogReg
    "021",          # TabM branch
    "026",          # LGB branch
    "028",          # structurally different NN branch
    "029",          # current main RealMLP representative
    "032",          # custom Torch RealMLP auxiliary branch
}

def build_pruned_model_keys(model_keys, hc_w, nm_w, signed_w):
    rows = []
    for k, wh, wn, ws in zip(model_keys, hc_w, nm_w, signed_w):
        max_w = max(float(wh), float(wn), float(ws))
        min_w = min(float(wh), float(wn), float(ws))

        if max_w < PRUNE_HARD_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_hard"
        elif max_w < PRUNE_SOFT_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_soft"
        else:
            decision = "keep"

        rows.append({
            "key": k,
            "hc_weight": float(wh),
            "nm_weight": float(wn),
            "slsqp_weight": float(ws),
            "max_weight": max_w,
            "min_weight": min_w,
            "protected": k in KEEP_PROTECTED_KEYS,
            "decision": decision,
        })

    prune_df = pd.DataFrame(rows)
    keep_keys = prune_df.loc[prune_df["decision"] == "keep", "key"].tolist()
    drop_keys = prune_df.loc[prune_df["decision"] != "keep", "key"].tolist()

    return keep_keys, drop_keys, prune_df


keep_keys, drop_keys, prune_df = build_pruned_model_keys(
    model_keys=model_keys,
    hc_w=hc_w,
    nm_w=nm_w,
    signed_w=signed_w,
)

print("keep_keys:", keep_keys)
print("drop_keys:", drop_keys)
display(prune_df)

prune_df.to_csv(CFG.OUTDIR / "prune_decision_after_hc_nm_slsqp.csv", index=False)

keep_keys: ['014', '015', '016', '017', '021', '026', '028', '029', '032']
drop_keys: []


,key,hc_weight,nm_weight,slsqp_weight,max_weight,min_weight,protected,decision
0,014,0.060647,0.059783,0.059782,0.060647,0.059782,True,keep
1,015,0.048587,0.049073,0.049195,0.049195,0.048587,True,keep
2,016,0.110392,0.111078,0.111133,0.111133,0.110392,True,keep
3,017,0.025640,0.024783,0.024554,0.025640,0.024554,True,keep
4,021,0.035954,0.036098,0.036138,0.036138,0.035954,True,keep
5,026,0.053879,0.053580,0.053614,0.053879,0.053580,True,keep
6,028,0.072194,0.071134,0.071095,0.072194,0.071095,True,keep
7,029,0.401839,0.403298,0.403409,0.403409,0.401839,True,keep
8,032,0.190868,0.191175,0.191078,0.191175,0.190868,True,keep


In [17]:
# ============================================================
# Rebuild matrices for pruned Ridge / LogReg
# ============================================================

pruned_idx = [model_keys.index(k) for k in keep_keys]

model_keys_full = model_keys.copy()
model_keys_pruned = keep_keys.copy()

X_raw_full = X_raw
T_raw_full = T_raw

X_raw_pruned = X_raw[:, pruned_idx]
T_raw_pruned = T_raw[:, pruned_idx]

# Temporarily switch global model_keys for meta feature names
model_keys = model_keys_pruned
X_meta, T_meta, meta_feature_names = build_meta_features(
    X_raw_pruned,
    T_raw_pruned,
    keep_keys,
)

print("Pruned X_raw:", X_raw_pruned.shape)
print("Pruned X_meta:", X_meta.shape)
print("Pruned model_keys:", model_keys)

Pruned X_raw: (439140, 9)
Pruned X_meta: (439140, 27)
Pruned model_keys: ['014', '015', '016', '017', '021', '026', '028', '029', '032']


In [18]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_pruned",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-4, 1e-1, 7)
    for l1 in [0.05, 0.10, 0.20, 0.50, 0.90]
]


def build_elastic_refined_grid_fast(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.5,
        factor_high=2.0,
        n=7,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = sorted(set([
        round(max(0.001, l1_center - 0.10), 6),
        round(l1_center, 6),
        round(min(0.999, l1_center + 0.10), 6),
        0.05,
        0.10,
        0.20,
        0.50,
        0.90,
    ]))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


# elastic_best, elastic_hist = run_meta_regressor_two_stage(
#     name="elasticnet_meta_all",
#     estimator_factory=lambda p: ElasticNet(
#         alpha=p["alpha"],
#         l1_ratio=p["l1_ratio"],
#         max_iter=30000,
#         random_state=CFG.SEED,
#         selection="cyclic",
#     ),
#     coarse_grid=elastic_coarse_grid,
#     refine_builder=build_elastic_refined_grid_fast,
# )


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_pruned",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_pruned coarse search
{'alpha': np.float64(0.001)} 0.9546184312830284
{'alpha': np.float64(0.0021544346900318843)} 0.9546184327145253
{'alpha': np.float64(0.004641588833612777)} 0.9546184338206822
{'alpha': np.float64(0.01)} 0.9546184385381156
{'alpha': np.float64(0.021544346900318832)} 0.9546184441990355
{'alpha': np.float64(0.046415888336127774)} 0.954618461767408
{'alpha': np.float64(0.1)} 0.9546184899744061
{'alpha': np.float64(0.21544346900318823)} 0.9546185706913176
{'alpha': np.float64(0.46415888336127775)} 0.9546187196320757
{'alpha': np.float64(1.0)} 0.9546190291933059
{'alpha': np.float64(2.154434690031882)} 0.954619640182261
{'alpha': np.float64(4.6415888336127775)} 0.9546207168957572
{'alpha': np.float64(10.0)} 0.9546223590505765
{'alpha': np.float64(21.54434690031882)} 0.9546244980975311
{'alpha': np.float64(46.41588833612773)} 0.9546269184336501
{'alpha': np.float64(100.0)} 0.9546282879858891
{'alpha': np.float64(21

,stage,score,alpha
29,refined,0.954628,100.000000
15,coarse,0.954628,100.000000
28,refined,0.954628,85.133992
27,refined,0.954628,72.477966
30,refined,0.954628,117.461894
26,refined,0.954628,61.703386
31,refined,0.954627,137.972966
25,refined,0.954627,52.530556
14,coarse,0.954627,46.415888
24,refined,0.954627,44.721360



logreg_meta_pruned coarse search
{'C': np.float64(0.001)} 0.9547468849437292
{'C': np.float64(0.0021544346900318843)} 0.9547662838755828
{'C': np.float64(0.004641588833612777)} 0.9547721472223047
{'C': np.float64(0.01)} 0.9547749481414266
{'C': np.float64(0.021544346900318832)} 0.9547737490674691
{'C': np.float64(0.046415888336127774)} 0.9547714678468331
{'C': np.float64(0.1)} 0.9547694409771903
{'C': np.float64(0.21544346900318823)} 0.9547677154051359
{'C': np.float64(0.46415888336127775)} 0.954768368883525
{'C': np.float64(1.0)} 0.9547673693407316
{'C': np.float64(2.154434690031882)} 0.954767839229628
{'C': np.float64(4.6415888336127775)} 0.9547673356354835
{'C': np.float64(10.0)} 0.9547679849169837
{'C': np.float64(21.54434690031882)} 0.9547665067336414
{'C': np.float64(46.41588833612773)} 0.9547666708027202
{'C': np.float64(100.0)} 0.9547671018785271
{'C': np.float64(215.44346900318823)} 0.954766809527795
{'C': np.float64(464.1588833612773)} 0.9547668781745838
{'C': np.float64(100

,stage,score,C
3,coarse,0.954775,0.010000
29,refined,0.954775,0.010000
30,refined,0.954774,0.011746
4,coarse,0.954774,0.021544
28,refined,0.954773,0.008513
34,refined,0.954773,0.022361
32,refined,0.954773,0.016207
25,refined,0.954773,0.005253
36,refined,0.954773,0.030852
33,refined,0.954773,0.019037


In [19]:
# ============================================================
# D additions: LogReg topK and feature-set variants
# ============================================================
#
# Purpose:
# - Absorb only the low-cost ideas from stacking_vibe_coding.
# - Do not introduce full multi-layer stacking.
#
# Added candidates:
# - logreg_meta_pruned_top3_equal / top5_equal from raw+rank+logit C-search
# - logreg variants: logit_only, raw_logit, raw_only
# - ridge variants: logit_only, raw_logit
# ============================================================

print("\n" + "=" * 80)
print("D additions: LogReg topK and feature-set variants")
print("=" * 80)


def fit_predict_logreg_meta_features(X_feat, T_feat, C):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_feat[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_feat[va_idx])[:, 1]

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_feat, y)
    test_meta = final_model.predict_proba(T_feat)[:, 1]

    return oof_meta, test_meta, auc(y, oof_meta)


def fit_predict_ridge_meta_features(X_feat, T_feat, alpha):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            Ridge(
                alpha=alpha,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_feat[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_feat[va_idx])

    final_model = make_pipeline(
        StandardScaler(),
        Ridge(
            alpha=alpha,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_feat, y)
    test_meta = final_model.predict(T_feat)

    return oof_meta, test_meta, auc(y, oof_meta)


def run_logreg_two_stage_on_feature_variant(variant):
    X_v, T_v, names_v = build_meta_features_variant(
        X_raw_pruned,
        T_raw_pruned,
        keep_keys,
        variant=variant,
    )

    print(f"\nLogReg feature variant: {variant}")
    print("X_v:", X_v.shape)

    coarse_grid = [{"C": c} for c in np.geomspace(1e-3, 1e3, 19)]

    history = []
    best = None

    for params in coarse_grid:
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_v, T_v, params["C"])
        history.append({"stage": "coarse", "score": float(score), **params})
        print("coarse", variant, params, score)
        if best is None or score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    refined_grid = [
        {"C": c}
        for c in make_refined_grid_1d(
            best["params"]["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ]

    for params in refined_grid:
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_v, T_v, params["C"])
        history.append({"stage": "refined", "score": float(score), **params})
        print("refined", variant, params, score)
        if score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    name = f"logreg_meta_pruned_{variant}"
    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=best["pred"],
        params={**best["params"], "feature_variant": variant},
        notes=f"D feature-set variant: LogisticRegression on {variant} meta features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)
    display(hist_df.head(20))

    return best, hist_df


def run_ridge_two_stage_on_feature_variant(variant):
    X_v, T_v, names_v = build_meta_features_variant(
        X_raw_pruned,
        T_raw_pruned,
        keep_keys,
        variant=variant,
    )

    print(f"\nRidge feature variant: {variant}")
    print("X_v:", X_v.shape)

    coarse_grid = [{"alpha": a} for a in np.geomspace(1e-3, 1e3, 19)]

    history = []
    best = None

    for params in coarse_grid:
        oof_meta, test_meta, score = fit_predict_ridge_meta_features(X_v, T_v, params["alpha"])
        history.append({"stage": "coarse", "score": float(score), **params})
        print("coarse", variant, params, score)
        if best is None or score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    refined_grid = [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best["params"]["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ]

    for params in refined_grid:
        oof_meta, test_meta, score = fit_predict_ridge_meta_features(X_v, T_v, params["alpha"])
        history.append({"stage": "refined", "score": float(score), **params})
        print("refined", variant, params, score)
        if score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    name = f"ridge_meta_pruned_{variant}"
    add_candidate(
        name=name,
        method="ridge",
        oof_pred=best["oof"],
        test_pred=best["pred"],
        params={**best["params"], "feature_variant": variant},
        notes=f"D feature-set variant: Ridge on {variant} meta features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)
    display(hist_df.head(20))

    return best, hist_df


# ----------------------------
# LogReg topK equal blend using the baseline raw+rank+logit search history
# ----------------------------

def add_logreg_topk_equal_from_history(k):
    hist = logreg_hist.sort_values("score", ascending=False).copy()
    top = hist.drop_duplicates(subset=["C"]).head(k)

    oofs_top = []
    preds_top = []
    rows = []

    for _, row in top.iterrows():
        C = float(row["C"])
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_meta, T_meta, C)
        oofs_top.append(oof_meta)
        preds_top.append(test_meta)
        rows.append({"C": C, "cv_auc_recomputed": float(score), "source_score": float(row["score"])})

    oof_avg = np.mean(oofs_top, axis=0)
    pred_avg = np.mean(preds_top, axis=0)

    name = f"logreg_meta_pruned_top{k}_equal"
    add_candidate(
        name=name,
        method="logreg_topk_equal",
        oof_pred=oof_avg,
        test_pred=pred_avg,
        params={
            "k": k,
            "feature_variant": "raw_rank_logit",
            "Cs": [float(r["C"]) for r in rows],
        },
        notes=f"D top{k} equal blend of LogisticRegression C candidates on raw+rank+logit features",
    )

    top_df = pd.DataFrame(rows)
    top_df.to_csv(CFG.OUTDIR / f"{name}_components.csv", index=False)
    display(top_df)

    return top_df


logreg_top3_components = add_logreg_topk_equal_from_history(3)
logreg_top5_components = add_logreg_topk_equal_from_history(5)


# ----------------------------
# LogReg feature-set variants
# ----------------------------

logreg_variant_results = {}
for variant in ["logit_only", "raw_logit", "raw_only"]:
    best_v, hist_v = run_logreg_two_stage_on_feature_variant(variant)
    logreg_variant_results[variant] = {
        "best": best_v,
        "history": hist_v,
    }


# ----------------------------
# Ridge feature-set variants
# ----------------------------

ridge_variant_results = {}
for variant in ["logit_only", "raw_logit"]:
    best_v, hist_v = run_ridge_two_stage_on_feature_variant(variant)
    ridge_variant_results[variant] = {
        "best": best_v,
        "history": hist_v,
    }



D additions: LogReg topK and feature-set variants
logreg_meta_pruned_top3_equal: 0.954774455


,C,cv_auc_recomputed,source_score
0,0.010000,0.954775,0.954775
1,0.011746,0.954774,0.954774
2,0.021544,0.954774,0.954774


logreg_meta_pruned_top5_equal: 0.954774264


,C,cv_auc_recomputed,source_score
0,0.010000,0.954775,0.954775
1,0.011746,0.954774,0.954774
2,0.021544,0.954774,0.954774
3,0.008513,0.954773,0.954773
4,0.022361,0.954773,0.954773



LogReg feature variant: logit_only
X_v: (439140, 9)
coarse logit_only {'C': np.float64(0.001)} 0.9546520702917511
coarse logit_only {'C': np.float64(0.0021544346900318843)} 0.9547020930441289
coarse logit_only {'C': np.float64(0.004641588833612777)} 0.9547379291405039
coarse logit_only {'C': np.float64(0.01)} 0.9547560947726074
coarse logit_only {'C': np.float64(0.021544346900318832)} 0.9547615715826003
coarse logit_only {'C': np.float64(0.046415888336127774)} 0.9547629535953701
coarse logit_only {'C': np.float64(0.1)} 0.9547628738544793
coarse logit_only {'C': np.float64(0.21544346900318823)} 0.9547624900831422
coarse logit_only {'C': np.float64(0.46415888336127775)} 0.9547610690295445
coarse logit_only {'C': np.float64(1.0)} 0.9547609434156812
coarse logit_only {'C': np.float64(2.154434690031882)} 0.954760871808296
coarse logit_only {'C': np.float64(4.6415888336127775)} 0.9547608401852257
coarse logit_only {'C': np.float64(10.0)} 0.9547608262606637
coarse logit_only {'C': np.float64

...

,stage,score,C
20,refined,0.954459,0.000235
19,refined,0.954456,0.000200
22,refined,0.954450,0.000324
21,refined,0.954449,0.000276
23,refined,0.954446,0.000381
24,refined,0.954441,0.000447
25,refined,0.954434,0.000525
26,refined,0.954429,0.000617
27,refined,0.954424,0.000725
28,refined,0.954423,0.000851



Ridge feature variant: logit_only
X_v: (439140, 9)
coarse logit_only {'alpha': np.float64(0.001)} 0.950799612598087
coarse logit_only {'alpha': np.float64(0.0021544346900318843)} 0.9507996133138356
coarse logit_only {'alpha': np.float64(0.004641588833612777)} 0.9507996157864214
coarse logit_only {'alpha': np.float64(0.01)} 0.9507996199507762
coarse logit_only {'alpha': np.float64(0.021544346900318832)} 0.9507996288650986
coarse logit_only {'alpha': np.float64(0.046415888336127774)} 0.950799647669764
coarse logit_only {'alpha': np.float64(0.1)} 0.9507996888253032
coarse logit_only {'alpha': np.float64(0.21544346900318823)} 0.9507997763743596
coarse logit_only {'alpha': np.float64(0.46415888336127775)} 0.9507999578816748
coarse logit_only {'alpha': np.float64(1.0)} 0.9508003609782218
coarse logit_only {'alpha': np.float64(2.154434690031882)} 0.9508012087497973
coarse logit_only {'alpha': np.float64(4.6415888336127775)} 0.9508030412611858
coarse logit_only {'alpha': np.float64(10.0)} 0.9

,stage,score,alpha
39,refined,0.952719,5000.000000
38,refined,0.952549,4256.699613
37,refined,0.952386,3623.898318
36,refined,0.952229,3085.169314
35,refined,0.952081,2626.527804
34,refined,0.951943,2236.067977
33,refined,0.951815,1903.653939
32,refined,0.951697,1620.656597
31,refined,0.951590,1379.729661
30,refined,0.951493,1174.618943



Ridge feature variant: raw_logit
X_v: (439140, 18)
coarse raw_logit {'alpha': np.float64(0.001)} 0.9544673961797943
coarse raw_logit {'alpha': np.float64(0.0021544346900318843)} 0.9544673958544541
coarse raw_logit {'alpha': np.float64(0.004641588833612777)} 0.9544673965376685
coarse raw_logit {'alpha': np.float64(0.01)} 0.9544673966678047
coarse raw_logit {'alpha': np.float64(0.021544346900318832)} 0.9544673970582129
coarse raw_logit {'alpha': np.float64(0.046415888336127774)} 0.9544673990427884
coarse raw_logit {'alpha': np.float64(0.1)} 0.9544674034999495
coarse raw_logit {'alpha': np.float64(0.21544346900318823)} 0.9544674122841358
coarse raw_logit {'alpha': np.float64(0.46415888336127775)} 0.9544674319997537
coarse raw_logit {'alpha': np.float64(1.0)} 0.9544674740011777
coarse raw_logit {'alpha': np.float64(2.154434690031882)} 0.9544675760929423
coarse raw_logit {'alpha': np.float64(4.6415888336127775)} 0.9544677492390137
coarse raw_logit {'alpha': np.float64(10.0)} 0.954468173612

,stage,score,alpha
31,refined,0.954486,640.413779
30,refined,0.954486,545.209817
29,refined,0.954486,464.158883
17,coarse,0.954486,464.158883
32,refined,0.954486,752.242156
28,refined,0.954485,395.156988
33,refined,0.954484,883.597887
27,refined,0.954483,336.412919
26,refined,0.954482,286.401749
18,coarse,0.954482,1000.000000


In [20]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    weight_keys = rec.get("weight_keys") or model_keys
    for k, w in zip(weight_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_029,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
2,nm_softmax_raw,nm,0.954809,0.000767,0.000767,"[0.05978283, 0.04907271, 0.11107765, 0.0247825...","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000172,0.996173,0.000201,0.996882
3,slsqp_signed_raw_neg005,slsqp_signed,0.954809,0.000767,0.000767,"[0.0597821, 0.04919519, 0.11113338, 0.02455421...","{""constraint"": ""sum1_signed"", ""neg_bound"": -0....",signed SLSQP on raw OOF predictions; high over...,0.000172,0.996173,0.000200,0.996882
1,hc_nonnegative_raw,hc,0.954809,0.000767,0.000767,"[0.0606472, 0.04858736, 0.11039198, 0.02563985...","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000172,0.996166,0.000201,0.996875
5,logreg_meta_pruned,logreg,0.954775,0.000733,0.000733,None,"{""C"": 0.01}",two-stage meta CV logistic regression on raw+r...,0.000049,0.988292,0.000051,0.989712
6,logreg_meta_pruned_top3_equal,logreg_topk_equal,0.954774,0.000732,0.000732,None,"{""k"": 3, ""feature_variant"": ""raw_rank_logit"", ...",D top3 equal blend of LogisticRegression C can...,0.000047,0.988673,0.000049,0.989978
7,logreg_meta_pruned_top5_equal,logreg_topk_equal,0.954774,0.000732,0.000732,None,"{""k"": 5, ""feature_variant"": ""raw_rank_logit"", ...",D top5 equal blend of LogisticRegression C can...,0.000048,0.988669,0.000049,0.990008
9,logreg_meta_pruned_raw_logit,logreg,0.954770,0.000728,0.000728,None,"{""C"": 0.3491598792543897, ""feature_variant"": ""...",D feature-set variant: LogisticRegression on r...,0.000031,0.996085,0.000040,0.996244
8,logreg_meta_pruned_logit_only,logreg,0.954764,0.000722,0.000722,None,"{""C"": 0.039515698779812425, ""feature_variant"":...",D feature-set variant: LogisticRegression on l...,0.000041,0.995622,0.000051,0.996017
4,ridge_meta_pruned,ridge,0.954628,0.000586,0.000586,None,"{""alpha"": 100.0}",two-stage meta CV on raw+rank+logit features,-0.061062,0.998819,-0.041040,0.993844
0,avg,avg,0.954510,0.000468,0.000468,"[0.11111111, 0.11111111, 0.11111111, 0.1111111...",{},"simple average of ['014', '015', '016', '017',...",0.000161,0.995791,0.000206,0.996009


,candidate,method,cv_auc,w_014,w_015,w_016,w_017,w_021,w_026,w_028,w_029,w_032
2,nm_softmax_raw,nm,0.954809,0.059783,0.049073,0.111078,0.024783,0.036098,0.053580,0.071134,0.403298,0.191175
3,slsqp_signed_raw_neg005,slsqp_signed,0.954809,0.059782,0.049195,0.111133,0.024554,0.036138,0.053614,0.071095,0.403409,0.191078
1,hc_nonnegative_raw,hc,0.954809,0.060647,0.048587,0.110392,0.025640,0.035954,0.053879,0.072194,0.401839,0.190868
0,avg,avg,0.954510,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111


In [21]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"sub_{safe_name}_{CFG.EXP_ID}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg 0.954509754615797 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_avg_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
hc_nonnegative_raw 0.9548090685235561 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_hc_nonnegative_raw_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
nm_softmax_raw 0.954809106295557 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_nm_softmax_raw_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
slsqp_signed_raw_neg005 0.9548090746724864 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_slsqp_signed_raw_neg005_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
ridge_meta_pruned 0.9546282879858891 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_ridge_meta_pruned_exp_20260516_033_blend_logreg_topk_feature_va

In [22]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-16",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys_full": model_keys_full if "model_keys_full" in globals() else model_keys,
        "model_keys_pruned": model_keys_pruned if "model_keys_pruned" in globals() else model_keys,
        "model_keys_current": model_keys,
        "n_models_full": len(model_keys_full) if "model_keys_full" in globals() else len(model_keys),
        "n_models_pruned": len(model_keys_pruned) if "model_keys_pruned" in globals() else len(model_keys),
        "D_additions": [
            "logreg_meta_pruned_top3_equal",
            "logreg_meta_pruned_top5_equal",
            "logreg_meta_pruned_logit_only",
            "logreg_meta_pruned_raw_logit",
            "logreg_meta_pruned_raw_only",
            "ridge_meta_pruned_logit_only",
            "ridge_meta_pruned_raw_logit"
        ],
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "D experiment: low-cost blend-code improvements only.",
        "Avg/HC/NM/SLSQP are raw full-stack methods before pruning.",
        "Ridge/LogReg baseline and variants are run on the pruned stack.",
        "Added LogReg top3/top5 equal blends and feature-set variants: logit_only, raw_logit, raw_only.",
        "Added Ridge feature-set variants: logit_only, raw_logit."
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.954809106295557, 'delta_vs_029': 0.0007670278234774841, 'delta_vs_best_single': 0.0007670278234774841, 'weights': '[0.05978283, 0.04907271, 0.11107765, 0.02478251, 0.03609762, 0.05358009, 0.07113406, 0.40329757, 0.19117496]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 190, "fun": -0.954809106295557}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 0.00017190117840649972, 'oof_max': 0.9961726406439307, 'pred_min': 0.00020050228760006277, 'pred_max': 0.9968816482795534}
